# K-Nearest Neighbors (KNN) for Image Recognition
### Classifying Images Based on Nearby Feature Representations

**Presented by (Team):** MD Azadur Rahman (4478), Rakib Hossain (4056), Aduri Akter (4032)  
**Course Teacher:** Ahsan Ullah, Assistant Professor, Dept. of CSE, World University of Bangladesh

---

## Learning Objectives

By the end of this notebook, students will be able to:
- Understand how digital images are converted into numerical **feature representations** (raw pixel vectors, engineered visual descriptors, and PCA embeddings).
- Calculate **Euclidean distances** between image feature vectors.
- Apply **KNN classification** step-by-step to classify an unknown (query) image.
- Explain the importance of **feature scaling** (`StandardScaler` / pixel normalization) and selecting an appropriate value of **$K$**.
- Implement, inspect, and visualize KNN image recognition in Python using `scikit-learn` on handwritten digit images.

## Introduction & Important Terminology

**K-Nearest Neighbors (KNN)** is a supervised machine learning algorithm based on a simple intuition: *data points with similar characteristics lie close to one another and likely belong to the same class.*

In **image recognition**, a computer does not "see" shapes the way humans do—it sees numbers. By extracting a **feature representation** (a numerical vector describing an image's pixels, edges, or learned patterns), we can place every image as a point in multi-dimensional feature space:
1. Store the feature vectors and known labels of all training images.
2. Extract the feature vector of a new (query) image.
3. Measure the distance between the query image and every training image.
4. Select the **$K$ nearest neighbor images**.
5. Assign the predicted label using **majority voting**.

### Important Terminology in Image KNN

| Term | Explanation in Image Recognition |
|---|---|
| **$K$** | Number of nearest training images considered when voting |
| **Feature Representation (Embedding)** | A numerical vector $[x_1, x_2, \dots, x_n]$ capturing an image's visual content |
| **Neighbor Image** | A labeled training image whose feature vector is close to the query image |
| **Euclidean Distance** | Straight-line distance between two image feature vectors in feature space |
| **Majority Voting** | Assigning the most frequent class label among the $K$ nearest images |
| **Training Image Dataset** | Previously labeled images stored as reference examples |

## How Images Become Feature Representations + Worked Example

There are two common ways to represent an image for KNN:
1. **Flattened Pixel Intensities:** An $8 \times 8$ grayscale image becomes a $64$-dimensional vector $[p_1, p_2, \dots, p_{64}]$.
2. **Extracted Feature Representations:** Compact numerical descriptors such as `[Center Brightness, Loop Score, Vertical Stroke Ratio]`, HOG descriptors, PCA components, or deep neural network embeddings.

---

### Worked Example: Classifying a Handwritten Digit Image (`Digit 0` vs. `Digit 1`)

**Problem:** We want to classify a new mystery handwritten digit image as **Digit 0** or **Digit 1** using 3 extracted visual features (each scored from $0$ to $10$):
- $x_1$: **Center Brightness** (Digit `0` is hollow in the center $\rightarrow$ low; Digit `1` passes through the center $\rightarrow$ high)
- $x_2$: **Loop / Contour Score** (Digit `0` forms a closed loop $\rightarrow$ high; Digit `1` is a single stroke $\rightarrow$ low)
- $x_3$: **Vertical Stroke Ratio** (how strongly pixels align along a central vertical axis)

#### Step 1: Store the Training Dataset

| Image | Center Brightness ($x_1$) | Loop Score ($x_2$) | Vertical Ratio ($x_3$) | Class Label ($Y$) |
|---|---|---|---|---|
| **Image A** | 2 | 8 | 3 | Digit 0 |
| **Image B** | 3 | 7 | 4 | Digit 0 |
| **Image C** | 8 | 2 | 9 | Digit 1 |
| **Image D** | 7 | 3 | 8 | Digit 1 |
| **Image E** | 9 | 1 | 9 | Digit 1 |

#### Step 2: Choose $K$ and Inspect the Mystery Image
- New mystery image feature vector: $X_{new} = (3, 6, 4)$
- Choose $K = 3$ (an odd number reduces the chance of a tie in binary classification).

#### Step 3: Calculate Euclidean Distances
Using the multi-feature Euclidean distance formula:
$$d(A, B) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

- $d(X_{new}, A) = \sqrt{(3-2)^2 + (6-8)^2 + (4-3)^2} = \sqrt{1 + 4 + 1} = \sqrt{6} \approx 2.45$
- $d(X_{new}, B) = \sqrt{(3-3)^2 + (6-7)^2 + (4-4)^2} = \sqrt{0 + 1 + 0} = \sqrt{1} = 1.00$
- $d(X_{new}, C) = \sqrt{(3-8)^2 + (6-2)^2 + (4-9)^2} = \sqrt{25 + 16 + 25} = \sqrt{66} \approx 8.12$
- $d(X_{new}, D) = \sqrt{(3-7)^2 + (6-3)^2 + (4-8)^2} = \sqrt{16 + 9 + 16} = \sqrt{41} \approx 6.40$
- $d(X_{new}, E) = \sqrt{(3-9)^2 + (6-1)^2 + (4-9)^2} = \sqrt{36 + 25 + 25} = \sqrt{86} \approx 9.27$

| Image | Feature Vector | Euclidean Distance from $X_{new}=(3,6,4)$ | Class Label |
|---|---|---|---|
| **B** | $(3, 7, 4)$ | **1.00** | **Digit 0** |
| **A** | $(2, 8, 3)$ | **2.45** | **Digit 0** |
| **D** | $(7, 3, 8)$ | **6.40** | **Digit 1** |
| **C** | $(8, 2, 9)$ | 8.12 | Digit 1 |
| **E** | $(9, 1, 9)$ | 9.27 | Digit 1 |

#### Step 4 & 5: Select the $K = 3$ Nearest Neighbors and Apply Majority Voting
- Top 3 nearest images: **B** (Digit 0), **A** (Digit 0), **D** (Digit 1).
- Votes: **2 votes for Digit 0**, **1 vote for Digit 1**.
- **Predicted Class:** **Digit 0**.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# Training image feature representations: [Center Brightness, Loop Score, Vertical Ratio]
image_names = ["Image A", "Image B", "Image C", "Image D", "Image E"]
X_train_ex = np.array([
    [2, 8, 3],  # Image A -> Digit 0
    [3, 7, 4],  # Image B -> Digit 0
    [8, 2, 9],  # Image C -> Digit 1
    [7, 3, 8],  # Image D -> Digit 1
    [9, 1, 9],  # Image E -> Digit 1
])
y_train_ex = np.array(["Digit 0", "Digit 0", "Digit 1", "Digit 1", "Digit 1"])

# Mystery query image feature vector
X_new = np.array([[3, 6, 4]])

# 1. Compute Euclidean distances manually to inspect every neighbor
distances_manual = np.sqrt(np.sum((X_train_ex - X_new) ** 2, axis=1))
print("Euclidean distances from mystery image (3, 6, 4):")
for name, vec, dist, label in sorted(zip(image_names, X_train_ex, distances_manual, y_train_ex), key=lambda x: x[2]):
    print(f"  {name} {vec.tolist()} -> distance = {dist:.2f} | label = {label}")

# 2. Verify with scikit-learn KNeighborsClassifier (K=3)
knn_ex = KNeighborsClassifier(n_neighbors=3)
knn_ex.fit(X_train_ex, y_train_ex)

dists, indices = knn_ex.kneighbors(X_new)
prediction = knn_ex.predict(X_new)[0]

print("\nTop K=3 Nearest Neighbors:")
for rank, (idx, dist) in enumerate(zip(indices[0], dists[0]), start=1):
    print(f"  #{rank}: {image_names[idx]} (distance = {dist:.2f}, class = {y_train_ex[idx]})")

print(f"\nPredicted Class for Mystery Image: {prediction}")

## Real-World Image Recognition on Handwritten Digits (`load_digits`)

Now let's scale from our 3-feature toy example to real image recognition using scikit-learn's **Handwritten Digits** dataset (`datasets.load_digits()`).

- **Total images:** $1,797$ grayscale images across **10 classes** (`0` through `9`).
- **Image resolution:** $8 \times 8$ pixels.
- **Raw pixel feature representation:** Each image is flattened into a vector of $8 \times 8 = 64$ numerical intensity features (values ranging from $0$ to $16$).

In [ ]:
import matplotlib.pyplot as plt
from sklearn import datasets

# Load the handwritten digits dataset
digits = datasets.load_digits()
X_images = digits.data       # Flattened 64-D feature vectors (1797, 64)
y_labels = digits.target     # True digit labels 0..9

print(f"Number of images: {X_images.shape[0]}")
print(f"Image grid shape: {digits.images[0].shape}")
print(f"Flattened feature vector length: {X_images.shape[1]} features per image")
print(f"Pixel intensity range: {X_images.min():.0f} to {X_images.max():.0f}")

# Visualize 1 sample image with pixel intensities + its 64-D feature vector
sample_idx = 0
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: 8x8 grayscale image with numeric pixel grid
img_2d = digits.images[sample_idx]
axes[0].imshow(img_2d, cmap="gray_r")
axes[0].set_title(f"8x8 Image Grid (Label: {y_labels[sample_idx]})", fontsize=12, fontweight="bold")
for r in range(8):
    for c in range(8):
        val = int(img_2d[r, c])
        axes[0].text(c, r, str(val), ha="center", va="center",
                     color="white" if val > 8 else "black", fontsize=8)
axes[0].set_xlabel("Pixel Column")
axes[0].set_ylabel("Pixel Row")

# Right: Flattened 64-D feature representation
axes[1].bar(range(64), X_images[sample_idx], color="#2563eb", alpha=0.85)
axes[1].set_title("Flattened 64-D Pixel Feature Representation", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Feature Index (0 to 63)")
axes[1].set_ylabel("Pixel Intensity (0 to 16)")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

# Gallery of one sample per class (0 to 9)
fig, axes = plt.subplots(1, 10, figsize=(14, 2.2))
for digit_class in range(10):
    idx = np.where(y_labels == digit_class)[0][0]
    axes[digit_class].imshow(digits.images[idx], cmap="gray_r")
    axes[digit_class].set_title(f"Class {digit_class}", fontsize=10)
    axes[digit_class].axis("off")
plt.suptitle("Sample Images Across All 10 Digit Classes", fontsize=12, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

## Feature Scaling in Image KNN

Because KNN computes distances directly between feature vectors, **features with larger numerical ranges can dominate the distance calculation**.

For example, suppose an image representation combines:
- Total pixel intensity sum (ranges from $0$ to $16,000$)
- Aspect ratio or edge symmetry (ranges from $0.0$ to $1.0$)

Without scaling, the $0$–$16,000$ feature completely overwhelms the $0$–$1$ feature—just as annual income ($10,000$–$100,000$) overwhelms age ($18$–$70$) in tabular KNN!

Even for pixel vectors, certain edge pixels rarely activate while center pixels have high variance. Using a `scikit-learn` **`Pipeline` with `StandardScaler()`** (or pixel normalization to $[0, 1]$) ensures every feature contributes fairly, and prevents data leakage when splitting train and test sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

# Split images into 75% training and 25% testing (stratified by digit class)
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X_images, y_labels, digits.images,
    test_size=0.25, random_state=42, stratify=y_labels
)

# Build a pipeline: StandardScaler -> KNeighborsClassifier(n_neighbors=3)
knn_pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=3, metric="euclidean")
)

# Fit (store training feature representations) and evaluate on test images
knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy (StandardScaler + KNN, K=3): {acc:.4f} ({acc * 100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", ax=ax)
ax.set_title("KNN Image Classification Confusion Matrix (K = 3)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Inspecting "Nearby Feature Representations" Visually

One of the greatest strengths of KNN is **interpretability**: we can inspect the exact training images whose feature representations are closest to a test query image!

Using `knn.kneighbors(X_query_scaled)`, we retrieve both:
1. The **indices** of the $K$ closest training images.
2. Their **Euclidean distances** in scaled feature space.

In [ ]:
# Extract scaler and KNN classifier from the pipeline
scaler = knn_pipeline.named_steps["standardscaler"]
knn_model = knn_pipeline.named_steps["kneighborsclassifier"]

# Pick 3 query images from the test set (including at least one misclassified image if any exist)
misclassified = np.where(y_pred != y_test)[0]
query_indices = [0, 15, int(misclassified[0])] if len(misclassified) > 0 else [0, 15, 30]

K_inspect = 5
X_query_scaled = scaler.transform(X_test[query_indices])
distances, neighbor_indices = knn_model.kneighbors(X_query_scaled, n_neighbors=K_inspect)

fig, axes = plt.subplots(len(query_indices), K_inspect + 1, figsize=(13, 2.6 * len(query_indices)))

for row_i, q_idx in enumerate(query_indices):
    # Column 0: Query image
    ax_q = axes[row_i, 0]
    ax_q.imshow(img_test[q_idx], cmap="Oranges")
    status = "Correct" if y_pred[q_idx] == y_test[q_idx] else "Misclassified"
    ax_q.set_title(f"QUERY\nTrue: {y_test[q_idx]} | Pred: {y_pred[q_idx]}\n({status})",
                   fontsize=9, fontweight="bold",
                   color="green" if status == "Correct" else "crimson")
    ax_q.axis("off")

    # Columns 1..K: Nearest training neighbors
    for col_j in range(K_inspect):
        nb_idx = neighbor_indices[row_i, col_j]
        nb_dist = distances[row_i, col_j]
        ax_nb = axes[row_i, col_j + 1]
        ax_nb.imshow(img_train[nb_idx], cmap="gray_r")
        ax_nb.set_title(f"Neighbor #{col_j + 1}\nLabel: {y_train[nb_idx]} | d={nb_dist:.2f}", fontsize=9)
        ax_nb.axis("off")

plt.suptitle("Query Images and Their K=5 Nearest Neighbor Training Images", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Compact Feature Representations: PCA Embeddings vs. Raw Pixels

Comparing raw pixels can struggle when images have slight shifts, noise, or thousands of dimensions (the *curse of dimensionality*).

Instead, modern image recognition extracts **compact feature representations** (such as **PCA embeddings**, HOG descriptors, or **CNN feature vectors**):
- Removes redundant pixel noise.
- Groups semantically similar images closer together in feature space.
- Makes nearest-neighbor distance search faster!

Below, we:
1. Project the images into a **2D PCA feature space** to visually see how nearby representations cluster by digit class.
2. Benchmark **accuracy and prediction speed** across three representations: Raw 64-D Pixels, Scaled 64-D Pixels, and a 20-D PCA Feature Representation.

In [ ]:
import time
from sklearn.decomposition import PCA

# 1. Visualize 2D PCA Feature Space and highlight a query point + its K=5 neighbors
pca_2d = make_pipeline(StandardScaler(), PCA(n_components=2, random_state=42))
X_train_2d = pca_2d.fit_transform(X_train)
X_test_2d = pca_2d.transform(X_test)

knn_2d = KNeighborsClassifier(n_neighbors=5).fit(X_train_2d, y_train)
sample_q = 5
dists_2d, idxs_2d = knn_2d.kneighbors(X_test_2d[sample_q:sample_q + 1])

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap="tab10", alpha=0.55, s=28)
plt.colorbar(scatter, ticks=range(10), label="Digit Class")

# Plot query point and dashed lines to its 5 nearest neighbors in 2D feature space
qx, qy = X_test_2d[sample_q]
plt.scatter(qx, qy, color="black", marker="*", s=260, zorder=5,
            label=f"Query Image (True: {y_test[sample_q]}, 2D Pred: {knn_2d.predict(X_test_2d[sample_q:sample_q + 1])[0]})")

for idx_nb in idxs_2d[0]:
    nx, ny = X_train_2d[idx_nb]
    plt.plot([qx, nx], [qy, ny], "k--", linewidth=1.3, alpha=0.8)
    plt.scatter(nx, ny, facecolors="none", edgecolors="black", s=110, linewidths=1.8, zorder=4)

plt.title("2D PCA Feature Space: Nearby Image Representations Cluster by Digit Class", fontsize=13, fontweight="bold")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(loc="best")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Compare Accuracy and Prediction Time across Feature Representations
representations = {
    "Raw 64-D Pixels": KNeighborsClassifier(n_neighbors=3),
    "Scaled 64-D Pixels": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "20-D PCA Feature Representation": make_pipeline(StandardScaler(), PCA(n_components=20, random_state=42), KNeighborsClassifier(n_neighbors=3)),
}

print(f"{'Representation':<34} | {'Test Accuracy':<14} | {'Predict Time (ms)':<18}")
print("-" * 72)
for name, model in representations.items():
    model.fit(X_train, y_train)
    t0 = time.perf_counter()
    preds = model.predict(X_test)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    acc_rep = accuracy_score(y_test, preds)
    print(f"{name:<34} | {acc_rep * 100:>11.2f}%  | {elapsed_ms:>14.2f} ms")

## Choosing the Value of $K$

Selecting $K$ directly controls the balance between **overfitting** and **underfitting**:
- **Small $K$ (e.g., $K = 1$):** Very sensitive to noisy or oddly written training images; may **overfit**.
- **Large $K$ (e.g., $K = 25$):** Smooths out noise by consulting many neighbors, but may cross class boundaries and **underfit**.
- **Optimal $K$:** Selected via validation data or cross-validation (and for binary classification, an **odd** $K$ helps avoid voting ties).

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 25]
train_accs = []
test_accs = []

for k in k_values:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    pipe.fit(X_train, y_train)
    train_accs.append(pipe.score(X_train, y_train))
    test_accs.append(pipe.score(X_test, y_test))

best_idx = int(np.argmax(test_accs))
best_k = k_values[best_idx]
best_acc = test_accs[best_idx]

plt.figure(figsize=(9, 4.8))
plt.plot(k_values, [a * 100 for a in train_accs], "o--", label="Training Accuracy", color="#64748b")
plt.plot(k_values, [a * 100 for a in test_accs], "s-", label="Test Accuracy", color="#2563eb", linewidth=2.2)
plt.scatter([best_k], [best_acc * 100], color="crimson", s=140, zorder=5,
            label=f"Best Test K = {best_k} ({best_acc * 100:.2f}%)")

plt.xticks(k_values)
plt.title("Impact of K on Image Recognition Accuracy", fontsize=13, fontweight="bold")
plt.xlabel("Number of Nearest Neighbors (K)")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## Advantages, Limitations & Practice Exercises

### Advantages & Limitations in Image Recognition

| Advantages | Limitations |
|---|---|
| Simple and intuitive (retrieves visually similar examples) | Prediction becomes slow as the image database grows |
| No explicit training phase; easy to add new image classes | Sensitive to feature scaling and irrelevant background pixels |
| Naturally supports multi-class recognition and nonlinear boundaries | High-dimensional raw pixels suffer from the *curse of dimensionality* |

### Real-World Applications
- **Handwritten Digit & OCR Recognition** (matching character crops to known templates).
- **Face Verification & Identity Lookup** (comparing 128-D or 512-D deep face embeddings).
- **Visual Product Search** (finding visually similar clothing or items from catalog embeddings).
- **Medical Image Retrieval** (locating similar X-ray or histopathology cases to assist diagnosis).

---

### Student Exercises
1. **Pen-and-Paper Check:** Using the worked example in Section 3 ($X_{new} = (3, 6, 4)$), what is the predicted digit when **$K = 1$** and when **$K = 5$**?
2. **Distance Metric Experiment:** Modify Cell 8 to use Manhattan distance (`metric="manhattan"` or `p=1`) and Cosine distance (`metric="cosine"`). Compare their test accuracy against Euclidean distance (`metric="euclidean"`).
3. **Distance-Weighted Voting:** Try `weights="distance"` in `KNeighborsClassifier` so closer neighbor images receive stronger votes than farther ones.